In [ ]:
!uv pip install duckdb pandas numpy matplotlib seaborn langgraph langchain-openai langchain-core openai python-dotenv
!uv pip install tabulate
!uv pip install fpdf2 markdown2 pillow
!uv pip install reportlab


In [ ]:
%pip install python-dotenv

In [ ]:
# Importações necessárias
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, START, END
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from opentelemetry.sdk._logs import ReadableLogRecord
import os

os.environ["OPENAI_API_KEY"] = "sk-"

llm = ChatOpenAI(model="gpt-4o", temperature=0)


In [4]:
# Criar um banco de dados fictício em memória
conn = duckdb.connect(database=':memory:')

# Criar dados fictícios mais robustos (2000 registros)
import numpy as np
import random
from datetime import datetime, timedelta

# Lista de produtos expandida
produtos = [
    'Smartphone Galaxy', 'Notebook Dell', 'Monitor LG', 'Teclado Mecânico', 
    'Mouse Gamer', 'Headset Bluetooth', 'Tablet iPad', 'Smartwatch Apple',
    'Câmera DSLR', 'Impressora HP', 'Roteador WiFi', 'SSD 1TB',
    'Memória RAM 16GB', 'Placa de Vídeo RTX', 'Processador Intel i7',
    'Caixa de Som JBL', 'Fone de Ouvido Sony', 'Carregador Portátil',
    'Cabo HDMI', 'Adaptador USB-C'
]

# Gerar 2000 registros
num_registros = 2000
start_date = datetime(2023, 1, 1)

# Criar dados
sales_data = pd.DataFrame({
    'order_id': range(1, num_registros + 1),
    'date': [start_date + timedelta(days=random.randint(0, 365)) for _ in range(num_registros)],
    'customer_id': [f'C{str(random.randint(1, 500)).zfill(3)}' for _ in range(num_registros)],
    'product': [random.choice(produtos) for _ in range(num_registros)],
    'quantity': np.random.randint(1, 20, size=num_registros),
    'price': np.random.uniform(10.0, 1000.0, size=num_registros),
    'region': [random.choice(['Norte', 'Sul', 'Leste', 'Oeste', 'Centro']) for _ in range(num_registros)],
    'payment_method': [random.choice(['Cartão de Crédito', 'Boleto', 'PIX', 'Transferência']) for _ in range(num_registros)]
})

# Adicionar coluna de valor total
sales_data['total_value'] = sales_data['quantity'] * sales_data['price']

# Criar a tabela no DuckDB
conn.execute("CREATE TABLE sales AS SELECT * FROM sales_data")

In [5]:
# Verificar o número de registros
result = conn.execute("SELECT COUNT(*) AS total_registros FROM sales").fetchall()
print(f"Total de registros na tabela: {result[0][0]}")

# Verificar a distribuição de produtos
product_distribution = conn.execute("""
    SELECT 
        product, 
        COUNT(*) as quantidade 
    FROM sales 
    GROUP BY product 
    ORDER BY quantidade DESC
""").df()

print("\nDistribuição de produtos:")
print(product_distribution)

Total de registros na tabela: 2000

Distribuição de produtos:
                 product  quantidade
0     Placa de Vídeo RTX         119
1              Cabo HDMI         108
2             Monitor LG         107
3    Fone de Ouvido Sony         104
4          Roteador WiFi         104
5       Teclado Mecânico         104
6       Caixa de Som JBL         102
7            Mouse Gamer         101
8          Notebook Dell         101
9            Câmera DSLR         101
10  Processador Intel i7         100
11      Smartwatch Apple          99
12       Adaptador USB-C          99
13     Headset Bluetooth          97
14   Carregador Portátil          97
15         Impressora HP          94
16      Memória RAM 16GB          93
17     Smartphone Galaxy          92
18           Tablet iPad          91
19               SSD 1TB          87


In [6]:
# Query para verificar o número total de registros
verification_query = """
SELECT 
    COUNT(*) AS total_registros,
    COUNT(DISTINCT product) AS total_produtos,
    MIN(date) AS data_mais_antiga,
    MAX(date) AS data_mais_recente,
    SUM(total_value) AS valor_total_vendas
FROM sales
"""

verification_result = conn.execute(verification_query).df()
print("\nVerificação do banco de dados:")
print(verification_result)

# Mostrar alguns registros de exemplo
sample_query = "SELECT * FROM sales LIMIT 10"
sample_result = conn.execute(sample_query).df()
print("\nExemplo de registros:")
print(sample_result)


Verificação do banco de dados:
   total_registros  total_produtos data_mais_antiga data_mais_recente  \
0             2000              20       2023-01-01        2024-01-01   

   valor_total_vendas  
0        1.027912e+07  

Exemplo de registros:
   order_id       date customer_id               product  quantity  \
0         1 2023-10-18        C021               SSD 1TB        14   
1         2 2023-03-11        C471   Fone de Ouvido Sony         5   
2         3 2023-10-31        C367  Processador Intel i7        16   
3         4 2023-05-21        C393           Câmera DSLR         7   
4         5 2023-05-08        C429      Caixa de Som JBL        11   
5         6 2023-02-15        C388           Tablet iPad        11   
6         7 2023-12-11        C066            Monitor LG         9   
7         8 2023-07-07        C160   Fone de Ouvido Sony        17   
8         9 2023-03-17        C306               SSD 1TB         1   
9        10 2023-03-15        C054         Impress

In [7]:
# Estruturas de dados
class State(TypedDict):
    user_request: str
    sql_query: str
    query_result: pd.DataFrame
    report: str
    visualization: Any
    visualizations: List[Any]
    visualization_notes: List[str]
    errors: List[str]

In [8]:
def parse_user_request(state: State) -> State:
    """
    Converte a solicitação do usuário em uma consulta SQL.
    """
    user_request = state['user_request']
    
    prompt = ChatPromptTemplate.from_template(
        """
Você é um assistente que cria consultas SQL com base em solicitações em linguagem natural.
A partir da solicitação abaixo, gere uma consulta SQL válida para o banco de dados fornecido.

Solicitação do usuário:
{user_request}

Tabelas disponíveis e seus esquemas:

Tabela 'sales':
- order_id (INTEGER)
- date (DATE)
- customer_id (VARCHAR)
- product (VARCHAR)
- quantity (INTEGER)
- price (FLOAT)
- region (VARCHAR)
- payment_method (VARCHAR)
- total_value (FLOAT)

Importante:
- Use apenas as tabelas e colunas fornecidas.
- Certifique-se de que a consulta seja compatível com o DuckDB.
- Retorne apenas a consulta SQL pura, sem marcadores markdown ou decorações.
- Para distribuições, agregue por categoria e use aliases claros: total_value para SUM(total_value),
  quantity para SUM(quantity), total_orders para COUNT(*) e average_price para AVG(price).
- Para séries mensais, retorne uma coluna month (DATE) e a métrica agregada.
- Métodos de pagamento utilizados: use COUNT(*) por payment_method para a distribuição de pedidos,
  salvo se a solicitação pedir explicitamente participação em valor.
- Não invente colunas. A tabela sales não possui uma coluna category.
- Não use ``` ou sql em sua resposta.
        """
    )
    
    response = llm.invoke(prompt.format(user_request=user_request))
    sql_query = response.content.strip()
    
    # Limpar marcadores markdown se ainda estiverem presentes
    sql_query = sql_query.replace('```sql', '').replace('```', '').strip()
    
    state['sql_query'] = sql_query
    return state

In [9]:
def execute_sql_query(state: State) -> State:
    """
    Executa a consulta SQL e armazena o resultado.
    """
    sql_query = state.get('sql_query', '')
    try:
        query_result = conn.execute(sql_query).df()
        state['query_result'] = query_result
    except Exception as e:
        state['errors'].append(f"Erro ao executar a consulta SQL: {e}")
    return state

In [10]:
from matplotlib.figure import Figure
from pandas.api.types import is_numeric_dtype, is_bool_dtype, is_datetime64_any_dtype
import re
import unicodedata
import textwrap

MAX_PIE_CATEGORIES = 8  # Até 7 categorias principais + Outros.


def _normalized_name(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(c for c in value if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


LABELS = {
    "product": "Produto", "region": "Região", "payment_method": "Método de pagamento",
    "category": "Categoria", "date": "Data", "month": "Mês", "year": "Ano",
    "total_value": "Valor total (R$)", "quantity": "Quantidade",
    "total_orders": "Quantidade de pedidos", "average_price": "Preço médio (R$)",
    "price": "Preço (R$)", "customer_id": "Cliente",
}


def _label(col):
    return LABELS.get(_normalized_name(col), str(col).replace("_", " ").capitalize())


def _additive_metric(col, query="", sql=""):
    """Pizza apenas para totais, quantidades, contagens ou participações não negativas."""
    name = _normalized_name(col)
    q = _normalized_name(query)
    nonadditive = r"(^|_)(avg|average|mean|median|media|medio|mediano|ratio|rate|taxa|min|max|distinct|unicos|unique|unitario)(_|$)"
    if re.search(nonadditive, name) or name == "price":
        return False
    if re.search(r"\b(?:avg|median|min|max)\s*\(", sql, re.I) or re.search(r"\bcount\s*\(\s*distinct\b", sql, re.I):
        return False
    if re.search(r"(^|_)(media|medio|medios|mediana|taxa|distintos|unicos)(_|$)", q):
        return False
    if name in {"order_id", "customer_id", "product_id", "id", "year", "month", "day"}:
        return False
    return bool(re.search(r"(^|_)(total|sum|count|quantity|quantidade|qtd|receita|revenue|faturamento|vendas|pedidos|orders|sales|frequencia|percent|percentage|percentual|percentagem|share|participacao|units|unidades)(_|$)", name))


def build_report_charts(df, query="", sql="", include_pie=True):
    """Retorna (lista de figuras, avisos); todas as proporções usam o resultado completo."""
    if df is None or df.empty:
        return [], ["Não há dados para visualização."]
    figures, notes = [], []
    numeric = [c for c in df.columns if is_numeric_dtype(df[c]) and not is_bool_dtype(df[c])
               and _normalized_name(c) not in {"order_id", "customer_id", "product_id", "id", "year", "month", "day"}]
    if not numeric:
        return [], ["Não há métrica numérica adequada para visualização."]
    priority = ["total_value", "quantity", "total_orders", "average_price", "price"]
    value_col = next((c for key in priority for c in numeric if _normalized_name(c) == key), numeric[0])
    dates = [c for c in df.columns if is_datetime64_any_dtype(df[c]) or _normalized_name(c) in {"date", "month", "year", "data", "mes", "ano", "period", "periodo"}]
    categories = [c for c in df.columns if c != value_col and c not in dates and not is_numeric_dtype(df[c])
                  and not is_bool_dtype(df[c]) and _normalized_name(c) not in {"order_id", "customer_id", "product_id", "id"}]
    query_norm = _normalized_name(query)
    if dates and (not categories or re.search(r"(^|_)(mensal|mensais|temporal|evolucao|tendencia|ano|mes)(_|$)", query_norm)):
        x_col, kind = dates[0], "line"
    elif categories:
        preferred = ["product", "region", "payment_method", "category"]
        x_col = next((c for key in preferred for c in categories if _normalized_name(c) == key), categories[0])
        kind = "bar"
        if len(categories) > 1 or dates:
            notes.append("O resultado possui mais de uma dimensão; gráficos de uma única distribuição foram omitidos para evitar somas ambíguas.")
            return [], notes
    else:
        candidates = [c for c in numeric if c != value_col]
        if not candidates or len(df) < 2:
            return [], ["O resultado escalar é apresentado na tabela, sem gráfico."]
        x_col, kind = candidates[0], "scatter"

    vals = pd.to_numeric(df[value_col], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    if not np.isfinite(vals).all():
        return [], ["A métrica contém valores ausentes ou não finitos; gráfico omitido."]
    metric, category = _label(value_col), _label(x_col)
    additive = _additive_metric(value_col, query, sql)
    if kind == "line":
        if _normalized_name(x_col) in {"year", "ano"} and is_numeric_dtype(df[x_col]):
            x = pd.to_datetime(df[x_col].astype(str), format="%Y", errors="coerce")
        elif _normalized_name(x_col) in {"month", "mes"} and is_numeric_dtype(df[x_col]):
            x = pd.to_numeric(df[x_col], errors="coerce")
        else:
            x = pd.to_datetime(df[x_col], errors="coerce")
        if x.isna().any():
            return [], ["Datas inválidas; gráfico temporal omitido."]
        series = pd.DataFrame({"x": x, "y": vals})
        if series.x.duplicated().any():
            if not additive:
                return [], ["Períodos duplicados com métrica não aditiva; agregue na consulta SQL."]
            series = series.groupby("x", as_index=False).y.sum()
        series = series.sort_values("x")
        fig = Figure(figsize=(8.4, 4.0), layout="constrained")
        ax = fig.subplots()
        ax.plot(series.x, series.y, marker="o" if len(series) <= 24 else None, linewidth=1.8)
        ax.set(title=f"Evolução de {metric}", xlabel=category, ylabel=metric)
        ax.grid(axis="y", alpha=.25)
        if is_datetime64_any_dtype(series.x):
            fig.autofmt_xdate(rotation=30)
        figures.append(("Evolução temporal", fig))
        return figures, notes
    if kind == "scatter":
        x = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
        if not np.isfinite(x).all():
            return [], ["Valores inválidos para dispersão."]
        fig = Figure(figsize=(8.4, 4.0), layout="constrained")
        ax = fig.subplots()
        ax.scatter(x, vals, alpha=.7)
        ax.set(title=f"{metric} × {category}", xlabel=category, ylabel=metric)
        figures.append(("Gráfico de dispersão", fig))
        return figures, notes

    labels = df[x_col].astype("string").fillna("(Sem informação)").astype(str)
    series = pd.Series(vals, index=labels.to_numpy(), dtype=float)
    if series.index.has_duplicates:
        if not additive:
            return [], ["Categorias repetidas com métrica não aditiva; calcule a agregação correta em SQL."]
        series = series.groupby(level=0, sort=False).sum()
    if not np.isfinite(series.to_numpy()).all():
        return [], ["Agregação numérica não finita; gráficos omitidos."]
    series = series.sort_values(ascending=False, kind="stable")
    top = series.head(12)
    fig = Figure(figsize=(8.4, max(3.4, min(6, .36*len(top)+1.0))), layout="constrained")
    ax = fig.subplots()
    ax.barh([textwrap.fill(str(x), 32) for x in top.index][::-1], top.to_numpy()[::-1])
    ax.set(title=f"{metric} por {category}", xlabel=metric)
    ax.grid(axis="x", alpha=.25)
    figures.append(("Gráfico de barras" + (" — 12 maiores categorias" if len(series)>12 else ""), fig))
    if not include_pie:
        return figures, notes
    if not additive:
        notes.append("Pizza omitida: a métrica não representa uma soma ou participação válida no total.")
        return figures, notes
    if (series < 0).any() or series.sum() <= 0:
        notes.append("Pizza omitida: valores negativos ou total igual a zero.")
        return figures, notes
    positive = series[series > 0]
    if len(positive) < 2:
        notes.append("Pizza omitida: menos de duas categorias com contribuição positiva.")
        return figures, notes
    pie = positive.copy()
    if len(pie) > MAX_PIE_CATEGORIES:
        rest = float(pie.iloc[MAX_PIE_CATEGORIES-1:].sum())
        pie = pie.iloc[:MAX_PIE_CATEGORIES-1].copy()
        other = "Outros (agrupados)"
        while other in positive.index:
            other += "*"
        pie.loc[other] = rest
    total = float(positive.sum())
    fig = Figure(figsize=(8.4, 4.5), layout="constrained")
    ax = fig.subplots()
    wedges, _ = ax.pie(pie.to_numpy(), startangle=90, counterclock=False,
                      wedgeprops={"edgecolor": "white", "linewidth": 1})
    legend = [f"{textwrap.fill(str(label), 32)} — {value/total:.1%}" for label, value in pie.items()]
    ax.legend(wedges, legend, loc="center left", bbox_to_anchor=(.99,.5), frameon=False, fontsize=9)
    ax.set_aspect("equal")
    ax.set_title(f"Participação de {metric} por {category}\nBase: resultado da consulta")
    figures.append(("Gráfico de pizza — participação no resultado", fig))
    if re.search(r"\b(?:limit|fetch\s+(?:first|next))\b", sql, re.I):
        notes.append("A pizza representa apenas os registros retornados pela consulta SQL (incluindo eventuais limites TOP/LIMIT), não necessariamente o total de vendas do banco.")
    return figures, notes


def generate_visualization(state: State) -> State:
    state['visualizations'] = []
    state['visualization_notes'] = []
    state['visualization'] = None
    try:
        figures, notes = build_report_charts(state.get('query_result'), state.get('user_request', ''), state.get('sql_query', ''))
        state['visualizations'] = figures
        state['visualization_notes'] = notes
        state['visualization'] = figures[0][1] if figures else None
    except Exception as exc:
        state['visualization_notes'].append(f"Visualização indisponível: {type(exc).__name__}: {exc}")
    return state


In [11]:
def generate_report(state: State) -> State:
    """
    Gera o relatório final com base nos dados e visualizações.
    """
    query_result = state.get('query_result')
    visualization = state.get('visualization')
    
    if query_result is None or query_result.empty:
        state['errors'].append("Nenhum dado disponível para gerar o relatório.")
        return state
    
    try:
        # Gerar um resumo dos dados
        num_rows = len(query_result)
        
        report = f"# Relatório Gerado com Base na Solicitação\n\n"
        report += f"**Solicitação original:** {state['user_request']}\n\n"
        report += f"**Consulta SQL executada:**\n```sql\n{state['sql_query']}\n```\n\n"
        
        report += f"## Resumo dos Dados\n\n"
        report += f"- Total de registros: {num_rows}\n"
        
        # Adicionar estatísticas específicas com base nas colunas disponíveis
        if 'total_value' in query_result.columns:
            total_value = query_result['total_value'].sum()
            avg_value = query_result['total_value'].mean()
            report += f"- Valor total: R$ {total_value:.2f}\n"
            report += f"- Valor médio: R$ {avg_value:.2f}\n"
        
        if 'product' in query_result.columns and query_result['product'].nunique() < 10:
            report += f"- Produtos incluídos: {', '.join(query_result['product'].unique())}\n"
        
        if 'region' in query_result.columns and query_result['region'].nunique() < 10:
            report += f"- Regiões incluídas: {', '.join(query_result['region'].unique())}\n"
        
        # Adicionar tabela de dados
        report += f"\n## Dados Detalhados\n\n"
        
        # Limitar a 10 linhas para não sobrecarregar o relatório
        sample_data = query_result.head(10)
        report += sample_data.to_markdown() + "\n"
        
        if len(query_result) > 10:
            report += f"\n*Mostrando 10 de {len(query_result)} registros*\n"
        
        state['report'] = report
    except Exception as e:
        state['errors'].append(f"Erro ao gerar relatório: {str(e)}")
    
    return state

In [12]:
#### TEstando a ferramenta com erros 
# Inicializar o grafo de estado
workflow = StateGraph(State)

# Adicionar nós
workflow.add_node("parse_user_request", parse_user_request)
workflow.add_node("execute_sql_query", execute_sql_query)
workflow.add_node("generate_visualization", generate_visualization)
workflow.add_node("generate_report", generate_report)

# Definir as arestas
workflow.add_edge(START, "parse_user_request")
workflow.add_edge("parse_user_request", "execute_sql_query")
workflow.add_edge("execute_sql_query", "generate_visualization")
workflow.add_edge("generate_visualization", "generate_report")
workflow.add_edge("generate_report", END)

# Compilar o grafo
app = workflow.compile()

# Visualizar o grafo (opcional)
from IPython.display import Image, display
# display(Image(app.get_graph().draw_mermaid_png()))

In [13]:
def generate_business_report(user_request: str):
    """
    Gera um relatório empresarial com base na solicitação do usuário.
    
    Args:
        user_request (str): Solicitação em linguagem natural.
    
    Returns:
        State: Estado final após o processamento.
    """
    initial_state = State(
        user_request=user_request,
        sql_query='',
        query_result=None,
        report='',
        visualization=None,
        visualizations=[],
        visualization_notes=[],
        errors=[]
    )
    final_state = app.invoke(initial_state)
    return final_state

# Teste com um exemplo
user_request = "Mostre o total de vendas por produto, ordenado do maior para o menor valor"

# Executar o fluxo
final_state = generate_business_report(user_request)

# Verificar o resultado
print("SQL Query gerada:")
print(final_state['sql_query'])
print("\n" + "-"*50 + "\n")

if final_state['errors']:
    print("Erros encontrados:")
    for error in final_state['errors']:
        print(f"- {error}")
else:
    print("Relatório:")
    print(final_state['report'])
    
    for name, figure in final_state.get('visualizations', []):
        print(name)
        display(figure)
    for note in final_state.get('visualization_notes', []):
        print("Nota:", note)


SQL Query gerada:
SELECT product, SUM(total_value) AS total_value
FROM sales
GROUP BY product
ORDER BY total_value DESC;

--------------------------------------------------

Relatório:
# Relatório Gerado com Base na Solicitação

**Solicitação original:** Mostre o total de vendas por produto, ordenado do maior para o menor valor

**Consulta SQL executada:**
```sql
SELECT product, SUM(total_value) AS total_value
FROM sales
GROUP BY product
ORDER BY total_value DESC;
```

## Resumo dos Dados

- Total de registros: 20
- Valor total: R$ 10279124.05
- Valor médio: R$ 513956.20

## Dados Detalhados

|    | product              |   total_value |
|---:|:---------------------|--------------:|
|  0 | Placa de Vídeo RTX   |        624678 |
|  1 | Processador Intel i7 |        593856 |
|  2 | Notebook Dell        |        593796 |
|  3 | Monitor LG           |        578379 |
|  4 | Smartwatch Apple     |        561818 |
|  5 | Fone de Ouvido Sony  |        553408 |
|  6 | Teclado Mecânico     |   

<Figure size 840x532 with 1 Axes>

Gráfico de pizza — participação no resultado


<Figure size 840x450 with 1 Axes>

In [14]:
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, KeepTogether
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Image as ReportLabImage
from xml.sax.saxutils import escape
from pathlib import Path
import io
import re


def generate_multi_query_report(queries, filename="relatorio_multiplo.pdf",
                                title="Relatório Analítico Consolidado", *, include_pie=True):
    """Gera PDF com barras e pizza para distribuições válidas, e linha para séries temporais."""
    doc = SimpleDocTemplate(str(filename), pagesize=letter, leftMargin=42, rightMargin=42,
                            topMargin=42, bottomMargin=42)
    styles = getSampleStyleSheet()
    styles['Title'].alignment = 1
    normal = styles['Normal']
    heading1 = styles['Heading1']
    heading2 = styles['Heading2']
    cell_style = normal.clone('TableCell')
    cell_style.fontSize = 8
    cell_style.leading = 10
    cell_style.splitLongWords = 1
    elements = [Paragraph(escape(str(title)), styles['Title']), Spacer(1, 16)]
    final_states = []
    for i, query in enumerate(queries):
        print(f"Processando consulta {i+1}/{len(queries)}: {query}")
        if i:
            elements.append(PageBreak())
        elements.append(Paragraph(f"Análise {i+1}: {escape(str(query))}", heading1))
        elements.append(Spacer(1, 6))
        try:
            state = generate_business_report(query)
        except Exception as exc:
            state = {'user_request': query, 'errors': [f'{type(exc).__name__}: {exc}']}
        final_states.append(state)
        if state.get('errors'):
            for err in state['errors']:
                print(f"- {err}")
                elements.append(Paragraph(escape(str(err)), normal))
            continue
        report = state.get('report', '')
        sql = state.get('sql_query', '')
        if sql:
            elements.append(Paragraph('<b>Consulta SQL executada:</b>', normal))
            elements.append(Paragraph(f"<font face='Courier'>{escape(sql)}</font>", normal))
            elements.append(Spacer(1, 8))
        elements.append(Paragraph('Resumo dos dados', heading2))
        summary = re.search(r'## Resumo dos Dados\n\n(.*?)(?=\n\n##|\Z)', report, re.DOTALL)
        if summary:
            for item in re.findall(r'^- (.*)$', summary.group(1), re.MULTILINE):
                elements.append(Paragraph('• ' + escape(item), normal))
        df = state.get('query_result')
        if df is None or df.empty:
            elements.append(Paragraph('Sem dados para exibir.', normal))
            continue
        elements.append(Paragraph('Dados detalhados', heading2))
        sample = df.head(10)
        def cell(value):
            if isinstance(value, (float, np.floating)):
                value = f'{value:,.2f}'
            return Paragraph(escape(str(value)), cell_style)
        table_data = [[cell(c) for c in sample.columns]] + [[cell(x) for x in row] for row in sample.itertuples(index=False, name=None)]
        table = Table(table_data, repeatRows=1, colWidths=[(doc.width-12)/len(sample.columns)]*len(sample.columns))
        table.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.lightgrey),
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.whitesmoke]),
            ('GRID', (0,0), (-1,-1), .3, colors.lightgrey),
            ('TOPPADDING', (0,0), (-1,-1), 5), ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ]))
        elements.append(table)
        if len(df) > 10:
            elements.append(Paragraph(f'Mostrando 10 de {len(df)} registros. As proporções da pizza usam todo o resultado.', normal))
        # Reutiliza exatamente os gráficos do LangGraph; fallback para estados antigos.
        charts = state.get('visualizations')
        notes = list(state.get('visualization_notes', []))
        if charts is None:
            charts, notes = build_report_charts(df, query, sql, include_pie=include_pie)
        if not include_pie:
            charts = [(name, fig) for name, fig in charts if 'pizza' not in name.lower()]
            notes = [note for note in notes if 'pizza' not in note.lower()]
        for name, fig in charts:
            try:
                img_data = io.BytesIO()
                fig.savefig(img_data, format='png', dpi=180, facecolor='white')
                img_data.seek(0)
                img = ReportLabImage(img_data)
                # Preserva a proporção; a pizza permanece circular.
                scale = min((doc.width-12)/img.imageWidth, (4.25*inch)/img.imageHeight)
                img.drawWidth = img.imageWidth * scale
                img.drawHeight = img.imageHeight * scale
                elements.append(KeepTogether([Spacer(1, 9), Paragraph(escape(name), heading2), img]))
            except Exception as exc:
                notes.append(f'Erro ao adicionar gráfico {name}: {type(exc).__name__}: {exc}')
        for note in notes:
            elements.append(Paragraph(escape(str(note)), normal))
    doc.build(elements)
    print(f'Relatório múltiplo salvo como {Path(filename).resolve()}')
    return final_states


In [15]:
# Testes locais dos gráficos (sem chamadas ao LLM ou a APIs).
def test_report_charts():
    demo = pd.DataFrame({'region': [f'R{i}' for i in range(1, 13)], 'total_value': np.arange(1, 13, dtype=float)})
    charts, notes = build_report_charts(demo, sql='SELECT region, SUM(total_value) AS total_value FROM sales GROUP BY region')
    assert len(charts) == 2 and 'pizza' in charts[1][0]
    wedges = charts[1][1].axes[0].patches
    assert len(wedges) == MAX_PIE_CATEGORIES
    assert abs(sum(w.theta2 - w.theta1 for w in wedges) - 360) < .001
    labels = charts[1][1].axes[0].get_legend().get_texts()
    assert any('Outros (agrupados)' in t.get_text() for t in labels)
    grouped = pd.DataFrame({'region': ['Norte','Norte','Sul'], 'quantity': [2,3,5]})
    charts, _ = build_report_charts(grouped)
    assert len(charts) == 2 and len(charts[1][1].axes[0].patches) == 2
    assert all(abs(w.theta2-w.theta1-180) < .001 for w in charts[1][1].axes[0].patches)
    for values in ([0,0], [-1,3], [1,np.nan]):
        charts, _ = build_report_charts(pd.DataFrame({'region':['A','B'], 'quantity':values}))
        assert not any('pizza' in name for name,_ in charts)
    charts, _ = build_report_charts(pd.DataFrame({'product':['A','B'], 'average_price':[20,40]}))
    assert len(charts)==1 and 'barras' in charts[0][0]
    charts, _ = build_report_charts(pd.DataFrame({'month':pd.date_range('2023-01-01', periods=3, freq='MS'), 'total_value':[1.,2.,3.]}))
    assert len(charts)==1 and 'temporal' in charts[0][0]
    charts, _ = build_report_charts(demo, include_pie=False)
    assert len(charts)==1
    print('OK: barras, pizza, agrupamento, percentuais, dados inválidos, médias e séries temporais.')

test_report_charts()

OK: barras, pizza, agrupamento, percentuais, dados inválidos, médias e séries temporais.


In [16]:
# Lista de consultas para incluir no relatório.
queries = [
    "Quais são os 5 produtos mais vendidos em termos de valor total?",
    "Como se comparam as vendas totais entre as diferentes regiões?",
    "Qual é a distribuição dos pedidos por método de pagamento? Apresente em barras e pizza.",
    "Mostre a evolução das vendas mensais ao longo de 2023"
]

resultados = generate_multi_query_report(
    queries, "relatorio_consolidado.pdf", "Análise Completa de Vendas 2023",
    include_pie=True,
)


Processando consulta 1/4: Quais são os 5 produtos mais vendidos em termos de valor total?
Processando consulta 2/4: Como se comparam as vendas totais entre as diferentes regiões?
Processando consulta 3/4: Qual é a distribuição dos pedidos por método de pagamento? Apresente em barras e pizza.
Processando consulta 4/4: Mostre a evolução das vendas mensais ao longo de 2023
Relatório múltiplo salvo como C:\Users\eugen\Desktop\Kiro\relatorio_consolidado.pdf


In [17]:
queries = [
    "Qual a quantidade vendida por produto?",
    "Qual o preço médio por produto?",
    "Qual a quantidade total de clientes distintos?",
    "Quais são os produtos com maior preço médio?",
    "Qual a quantidade de pedidos por região?"
]

resultados = generate_multi_query_report(
    queries, filename="relatorio_multiplo.pdf",
    title="Relatório Analítico de E-commerce", include_pie=True,
)


Processando consulta 1/5: Qual a quantidade vendida por produto?
Processando consulta 2/5: Qual o preço médio por produto?
Processando consulta 3/5: Qual a quantidade total de clientes distintos?
Processando consulta 4/5: Quais são os produtos com maior preço médio?
Processando consulta 5/5: Qual a quantidade de pedidos por região?
Relatório múltiplo salvo como C:\Users\eugen\Desktop\Kiro\relatorio_multiplo.pdf


In [18]:
from IPython.display import display, IFrame, FileLink
from pathlib import Path

arquivo = Path("relatorio_multiplo.pdf")

if arquivo.exists() and arquivo.stat().st_size > 0:
    print("PDF gerado:", arquivo.resolve())

    display(FileLink(str(arquivo)))

    display(
        IFrame(
            src=str(arquivo),
            width=900,
            height=600
        )
    )
else:
    print("O PDF não foi gerado.")

PDF gerado: C:\Users\eugen\Desktop\Kiro\relatorio_multiplo.pdf


c:\Users\eugen\Desktop\Kiro\relatorio_multiplo.pdf